# Atividade 2 — Previsão do valor de imóveis na Califórnia
**RNA e Deep Learning · Regressão linear múltipla com PyTorch**

Quanto a informação sobre renda, moradias e localização melhora a previsão em relação a usar apenas o preço médio? Cada observação representa um grupo de quarteirões do censo de 1990. O alvo é o valor mediano das moradias desse grupo, em dólares da época.

### Variáveis do California Housing

| Variável | Significado |
|---|---|
| MedInc | Renda mediana dos domicílios, em dezenas de milhares de dólares |
| HouseAge | Idade mediana das moradias, em anos |
| AveRooms | Número médio de cômodos por domicílio |
| AveBedrms | Número médio de quartos por domicílio |
| Population | Número de habitantes no grupo |
| AveOccup | Número médio de moradores por domicílio |
| Latitude | Latitude do grupo, em graus |
| Longitude | Longitude do grupo, em graus |
| MedHouseVal | Valor mediano das moradias, fornecido em unidades de US$ 100.000 |

[Descrição da base no scikit-learn](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from IPython.display import display

semente = 42
torch.manual_seed(semente)
torch.set_num_threads(1)

base = fetch_california_housing(as_frame=True)
X = base.data
y = (base.target * 100_000).rename('preco_usd')
print(f'{len(X)} grupos de quarteirões e {X.shape[1]} variáveis de entrada.')

ModuleNotFoundError: No module named 'sklearn'

## Divisão dos dados e análise exploratória
Foi reservado 30% das linhas e foi dividido essa reserva ao meio como pedido no anunciado da questão. As estatísticas e os gráficos exploratórios abaixo usam somente o treino. A validação serve para escolher a época, e o teste fica reservado para a comparação final.

In [ ]:
X_tr, X_res, y_tr, y_res = train_test_split(X, y, test_size=0.30, random_state=semente)
X_val, X_te, y_val, y_te = train_test_split(X_res, y_res, test_size=0.50, random_state=semente)
for nome, parte in [('Treino', X_tr), ('Validação', X_val), ('Teste', X_te)]:
    print(f'{nome}: {len(parte)} observações ({len(parte)/len(X):.0%})')
display(X_tr.join(y_tr).describe().T.round(2))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for eixo, coluna, rotulo in zip(eixos, ['MedInc', 'HouseAge'],
                                 ['Renda mediana (US$ 10 mil)', 'Idade mediana (anos)']):
    eixo.scatter(X_tr[coluna], y_tr / 1000, s=7, alpha=0.12, color='#236A87')
    eixo.set_xlabel(rotulo)
eixos[0].set_ylabel('Valor mediano (mil US$)')
fig.suptitle('Renda, idade das moradias e preço no conjunto de treino')
fig.tight_layout()
plt.show()

O preço tende a subir com a renda mediana, embora grupos com renda semelhante apresentem preços diferentes. A idade, isoladamente, mostra bastante dispersão. A concentração horizontal próxima de US$ 500 mil reflete o limite superior registrado para o alvo e dificulta a distinção entre as áreas mais caras.

## Padronização
Cada coluna de entrada ganha sua própria média e desvio padrão. Um segundo transformador padroniza o preço e permite recuperar dólares depois da previsão. Ambos são ajustados exclusivamente com o treino.

In [ ]:
x_scaler = StandardScaler().fit(X_tr)
y_scaler = StandardScaler().fit(y_tr.to_frame())

def tensor(array):
    return torch.tensor(array, dtype=torch.float32)

x_train = tensor(x_scaler.transform(X_tr))
x_val = tensor(x_scaler.transform(X_val))
y_train = tensor(y_scaler.transform(y_tr.to_frame()))
y_val_std = tensor(y_scaler.transform(y_val.to_frame()))

## Treinamento e escolha da época
A camada linear aprende oito pesos e um intercepto. Cada época aplica um passo de SGD com o conjunto de treino inteiro. Depois do passo, medimos o MSE de treino e de validação com os mesmos parâmetros. Foi guardado os pesos da época com menor MSE de validação.

In [ ]:
torch.manual_seed(semente)
model = nn.Linear(8, 1)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)
n_epochs = 1500
historico = []
melhor_custo = float('inf')

for epoca in range(1, n_epochs + 1):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(x_train), y_train)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        custo_tr = criterion(model(x_train), y_train).item()
        custo_val = criterion(model(x_val), y_val_std).item()
    historico.append((epoca, custo_tr, custo_val))
    if custo_val < melhor_custo:
        melhor_custo = custo_val
        melhor_epoca = epoca
        melhores_pesos = {k: v.clone() for k, v in model.state_dict().items()}

model.load_state_dict(melhores_pesos)
curvas = pd.DataFrame(historico, columns=['Época', 'Treino', 'Validação'])
ax = curvas.plot(x='Época', y=['Treino', 'Validação'], figsize=(9, 4))
ax.scatter(melhor_epoca, melhor_custo, color='black', label='Menor MSE de validação', zorder=3)
ax.set(title='Evolução do erro durante o treinamento', ylabel='MSE do preço padronizado')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Época selecionada: {melhor_epoca} | MSE de validação: {melhor_custo:.4f}')

## RMSE no teste, em dólares
Comparamos duas previsões nas mesmas observações. O baseline atribui a todos os grupos a média dos preços de treino. Para o modelo, é desfeita a padronização antes de calcular os erros. O RMSE é a raiz da média dos erros ao quadrado e dá maior peso a erros grandes.

In [ ]:
model.eval()
with torch.no_grad():
    pred_std = model(tensor(x_scaler.transform(X_te))).numpy()
pred_usd = y_scaler.inverse_transform(pred_std).ravel()
real_usd = y_te.to_numpy()
base_usd = np.full(real_usd.shape, y_tr.mean())

def rmse(real, previsto):
    return np.sqrt(np.mean((real - previsto) ** 2))

erro_modelo = rmse(real_usd, pred_usd)
erro_base = rmse(real_usd, base_usd)
ganho = 100 * (1 - erro_modelo / erro_base)
display(pd.Series({'Média do treino': erro_base, 'Regressão linear': erro_modelo},
                  name='RMSE no teste (US$)').round(2).to_frame())
print(f'Redução relativa do RMSE: {ganho:.1f}%')

## Interpretação dos oito pesos
Como todas as entradas foram padronizadas, um aumento de uma unidade em qualquer entrada representa um desvio padrão daquela variável no treino. Mantendo as outras sete fixas, o peso indica a variação prevista em desvios padrão do preço. Multiplicar o peso pelo desvio padrão do preço converte essa variação em dólares. Variáveis correlacionadas podem influenciar o sinal dos coeficientes, por isso a interpretação descreve associações condicionais.

In [ ]:
pesos = pd.Series(model.weight.detach().numpy().ravel(), index=X.columns, name='Peso')
efeitos = pesos.to_frame()
efeitos['Variação prevista (US$ por +1 desvio padrão)'] = pesos * y_scaler.scale_[0]
display(efeitos.sort_values('Peso', ascending=False).round(4))
print('Aumentam a previsão, do maior efeito ao menor:', ', '.join(pesos[pesos > 0].sort_values(ascending=False).index))
print('Diminuem a previsão, do maior efeito ao menor:', ', '.join(pesos[pesos < 0].sort_values().index))
print(f'Maior aumento: {pesos.idxmax()}. Maior redução: {pesos.idxmin()}.')

## Conclusão

In [ ]:
print(f'No teste, o RMSE da regressão foi US$ {erro_modelo:,.0f}, ante US$ {erro_base:,.0f} '
      f'do baseline. A redução foi de {ganho:.1f}%.')

O modelo linear serve como primeira referência para comparar regiões. Para usá-lo em uma decisão de compra, seria preciso definir uma tolerância de erro em dólares e considerar que a base descreve regiões da Califórnia em 1990. Seria interessante testar uma interação entre renda e localização, pois o efeito da renda pode variar entre regiões. 